# Version 8.0
- Entity to cluster comparison

In [1]:
import faiss
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfTransformer
from collections import defaultdict
from itertools import combinations
from sentence_transformers import SentenceTransformer, util
from fuzzywuzzy import fuzz

/Users/user/miniconda3/envs/graphmaker/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
class ClusterER:
    def __init__(self, pred2idx=None, sim_threshold=0.65, embedding_model="all-mpnet-base-v2"):
        """
        pred2idx: mapping predicate -> index for relational signatures
        sim_threshold: threshold to merge with existing cluster
        """
        self.model = SentenceTransformer(embedding_model)
        self.sim_threshold = sim_threshold

        # FAISS for cluster embeddings
        self.dim = 768  # mpnet embedding size
        self.faiss_index = faiss.IndexFlatIP(self.dim)
        self.int2cluster = {}  # mapping FAISS row -> cluster_id
        self.cluster_counter = 1

        # Clusters
        self.clusters = {}  # cluster_id -> cluster dict

        # Relational signature
        self.pred2idx = pred2idx or {}
        self.num_predicates = len(self.pred2idx)

    # ----------------------
    # Helper: String similarity
    # ----------------------
    def _string_sim(self, a, b):
        return fuzz.token_sort_ratio(a, b) / 100

    # ----------------------
    # Helper: Type compatibility
    # ----------------------
    def _predict_type(self, entity_text, predicate=None):
        # Dummy placeholder: override with NER model if desired
        return "UNKNOWN"

    def _types_compatible(self, type1, type2):
        if type1 == "UNKNOWN" or type2 == "UNKNOWN":
            return True
        return type1 == type2

    # ----------------------
    # FAISS utilities
    # ----------------------
    def _add_to_faiss(self, cluster_id, emb):
        row = self.faiss_index.ntotal
        self.faiss_index.add(emb)
        self.int2cluster[row] = cluster_id

    def _semantic_blocking(self, entity_text, top_k=10):
        """Retrieve candidate clusters via FAISS"""
        if self.faiss_index.ntotal == 0:
            return []

        query_emb = self.model.encode(entity_text)
        query_emb = np.asarray(query_emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(query_emb)

        scores, indices = self.faiss_index.search(query_emb, top_k)
        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1 or score < self.sim_threshold:
                continue
            candidates.append(self.int2cluster[idx])
        return candidates

    # ----------------------
    # Relational signatures
    # ----------------------
    def _compute_candidate_relvec(self, predicate=None, other_entity_id=None):
        """Build candidate relational vector for current triple"""
        vec = np.zeros((1, self.num_predicates * 2), dtype=np.float32)
        if predicate in self.pred2idx:
            p_idx = self.pred2idx[predicate]
            vec[0, p_idx] += 1  # subject role
            if other_entity_id is not None:
                vec[0, p_idx + self.num_predicates] += 1  # object role
        return vec

    def _cluster_relational_similarity(self, cluster_id, candidate_vec=None):
        cluster_vec = self.clusters[cluster_id]["rel_signature"]
        if candidate_vec is None:
            candidate_vec = np.zeros_like(cluster_vec)

        # cosine similarity
        c_norm = cluster_vec / (np.linalg.norm(cluster_vec) + 1e-10)
        e_norm = candidate_vec / (np.linalg.norm(candidate_vec) + 1e-10)
        return float(np.dot(c_norm, e_norm.T))

    # ----------------------
    # Cluster management
    # ----------------------
    def _create_new_cluster(self, entity_text, predicate=None):
        cluster_id = f"C{self.cluster_counter}"
        self.cluster_counter += 1

        emb = self.model.encode(entity_text)
        emb = np.asarray(emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(emb)

        rel_vec = self._compute_candidate_relvec(predicate)

        self.clusters[cluster_id] = {
            "members": {entity_text},
            "centroid": emb,
            "type": self._predict_type(entity_text, predicate),
            "neighbors": set(),
            "rel_signature": rel_vec
        }

        self._add_to_faiss(cluster_id, emb)
        return cluster_id, "INSERT"

    def _update_cluster(self, cluster_id, new_entity_text, predicate=None):
        cluster = self.clusters[cluster_id]

        new_emb = self.model.encode(new_entity_text)
        new_emb = np.asarray(new_emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(new_emb)

        # Update centroid
        cluster["centroid"] = normalize(cluster["centroid"] + new_emb, norm="l2")
        cluster["members"].add(new_entity_text)

        # Update relational signature
        if predicate:
            candidate_vec = self._compute_candidate_relvec(predicate)
            cluster["rel_signature"] += candidate_vec

        # Rebuild FAISS for simplicity
        self._rebuild_faiss()

    def _rebuild_faiss(self):
        self.faiss_index.reset()
        self.int2cluster = {}
        for cid, c in self.clusters.items():
            self._add_to_faiss(cid, c["centroid"])

    # ----------------------
    # Cluster-level similarity
    # ----------------------
    def _cluster_similarity(self, entity_text, cluster_id, predicate=None, other_entity_id=None):
        cluster = self.clusters[cluster_id]

        # string similarity
        s_sim = max(self._string_sim(entity_text, m) for m in cluster["members"])

        # embedding similarity
        context = f"{entity_text} in relation {predicate}" if predicate else entity_text
        emb_query = self.model.encode(context)
        e_sim = util.cos_sim(emb_query, cluster["centroid"]).item()

        # type similarity
        new_type = self._predict_type(entity_text, predicate)
        p_sim = 1.0 if self._types_compatible(new_type, cluster["type"]) else 0.0

        # relational similarity
        candidate_relvec = self._compute_candidate_relvec(predicate, other_entity_id)
        r_sim = self._cluster_relational_similarity(cluster_id, candidate_relvec)

        # composite
        return 0.25 * s_sim + 0.40 * e_sim + 0.10 * p_sim + 0.25 * r_sim

    # ----------------------
    # Resolve entity
    # ----------------------
    def resolve_entity(self, entity_text, predicate=None, other_entity_id=None):
        candidates = self._semantic_blocking(entity_text)
        best_score = 0
        best_cluster = None

        for cluster_id in candidates:
            score = self._cluster_similarity(entity_text, cluster_id, predicate, other_entity_id)
            if score > best_score:
                best_score = score
                best_cluster = cluster_id

        if best_cluster and best_score >= self.sim_threshold:
            self._update_cluster(best_cluster, entity_text, predicate)
            return best_cluster, "MERGE"

        return self._create_new_cluster(entity_text, predicate)

    # ----------------------
    # Get entity -> cluster mapping
    # ----------------------
    def get_entity_mapping(self):
        mapping = {}
        for cid, cluster in self.clusters.items():
            for m in cluster["members"]:
                mapping[m] = cid
        return mapping

    # ----------------------
    # Evaluation functions
    # ----------------------
    @staticmethod
    def pairwise_links(items):
        if len(items) == 0:
            return set()
        if len(items) == 1:
            return {frozenset(items)}
        return set(frozenset([a, b]) for a, b in combinations(items, 2))

    @staticmethod
    def build_gold_entity_map(gold_clusters):
        gold_entity_map = defaultdict(set)
        for gold_cluster, entities in gold_clusters.items():
            for e in entities:
                gold_entity_map[e].add(gold_cluster)
        return gold_entity_map

    @staticmethod
    def evaluate_cluster_alignment(gold_clusters, predicted_clusters):
        gold_entity_map = ClusterER.build_gold_entity_map(gold_clusters)
        for pred_cluster_id, pred_entities in predicted_clusters.items():
            matched_gold = defaultdict(set)
            for e in pred_entities:
                golds = gold_entity_map.get(e, set())
                if not golds:
                    matched_gold["UNSEEN"].add(e)
                else:
                    for g in golds:
                        matched_gold[g].add(e)
            print("="*50)
            print(f"Predicted Cluster: {pred_cluster_id}")
            print(f"Entities: {pred_entities}")
            if len(matched_gold) == 1:
                print(f"Pure match with gold cluster: {next(iter(matched_gold))}")
            else:
                print("Mixed cluster matches:")
                for g, ents in matched_gold.items():
                    print(f"  - {g}: {ents}")

    def evaluate(self, toy_dataset):
        # build gold clusters
        gold_clusters = defaultdict(set)
        for t in toy_dataset:
            gold_clusters[t["subject_gold"]].add(t["subject"])
            gold_clusters[t["object_gold"]].add(t["object"])

        # predicted clusters
        pred_clusters = defaultdict(set)
        mapping = self.get_entity_mapping()
        for mention, cid in mapping.items():
            pred_clusters[cid].add(mention)

        # pairwise metrics
        gold_links = set()
        for entities in gold_clusters.values():
            gold_links.update(self.pairwise_links(entities))
        pred_links = set()
        for entities in pred_clusters.values():
            pred_links.update(self.pairwise_links(entities))

        tp = len(gold_links & pred_links)
        fp = len(pred_links - gold_links)
        fn = len(gold_links - pred_links)
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1 = 2 * precision * recall / (precision + recall + 1e-10)

        print("\n=== Evaluation Results ===")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1       : {f1:.4f}")

        print("\n=== Cluster Alignment ===")
        self.evaluate_cluster_alignment(gold_clusters, pred_clusters)
        return precision, recall, f1


In [3]:
from knowledge_graph_maker import Edge, Node

# list_of_edges = [

#     # =========================
#     # 1. Homonym & Lexical Collision
#     # =========================

#     Edge(
#         node_1=Node(
#             label="Organization",
#             name="Apple",
#             properties={"entity_type": "technology_company"}
#         ),
#         node_2=Node(
#             label="Product",
#             name="iPhone 15",
#             properties={}
#         ),
#         relationship="released",
#         metadata={
#             "summary": "Apple released the iPhone 15.",
#             "subject_gold": "APPLE_TECH",
#             "object_gold": "IPHONE_15",
#             "generated_at": "2025-10-30 17:34:07.282620"
#         },
#         order=0
#     ),

#     Edge(
#         node_1=Node(
#             label="Food",
#             name="Apple",
#             properties={"entity_type": "fruit"}
#         ),
#         node_2=Node(
#             label="Time",
#             name="September",
#             properties={}
#         ),
#         relationship="harvested in",
#         metadata={
#             "summary": "Apple is harvested in September.",
#             "subject_gold": "APPLE_FRUIT",
#             "object_gold": "MONTH_SEPTEMBER",
#             "generated_at": "2025-10-30 17:34:07.282620"
#         },
#         order=1
#     ),

#     Edge(
#         node_1=Node(
#             label="Food",
#             name="Apple",
#             properties={"entity_type": "fruit"}
#         ),
#         node_2=Node(
#             label="FoodCategory",
#             name="Fruit",
#             properties={}
#         ),
#         relationship="is a type of",
#         metadata={
#             "summary": "Apple is a type of fruit.",
#             "subject_gold": "APPLE_FRUIT",
#             "object_gold": "FRUIT",
#             "generated_at": "2025-10-30 17:34:07.282620"
#         },
#         order=2
#     ),

#     Edge(
#         node_1=Node(
#             label="Organization",
#             name="Apple Inc.",
#             properties={"entity_type": "technology_company"}
#         ),
#         node_2=Node(
#             label="Product",
#             name="MacBook",
#             properties={}
#         ),
#         relationship="sells",
#         metadata={
#             "summary": "Apple Inc. sells the MacBook.",
#             "subject_gold": "APPLE_TECH",
#             "object_gold": "MACBOOK",
#             "generated_at": "2025-10-30 17:34:07.282620"
#         },
#         order=3
#     )
# ]

list_of_edges = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    Edge(
        node_1=Node(label="Organization", name="Apple", properties={}),
        node_2=Node(label="Product", name="iPhone 15", properties={}),
        relationship="released",
        metadata={"subject_gold": "APPLE_TECH", "object_gold": "IPHONE_15"},
        order=0
    ),
    Edge(
        node_1=Node(label="Food", name="Apple", properties={}),
        node_2=Node(label="Time", name="September", properties={}),
        relationship="harvested in",
        metadata={"subject_gold": "APPLE_FRUIT", "object_gold": "MONTH_SEPTEMBER"},
        order=1
    ),
    Edge(
        node_1=Node(label="Food", name="Apple", properties={}),
        node_2=Node(label="FoodCategory", name="Fruit", properties={}),
        relationship="is a type of",
        metadata={"subject_gold": "APPLE_FRUIT", "object_gold": "FRUIT"},
        order=2
    ),
    Edge(
        node_1=Node(label="Organization", name="Apple Inc.", properties={}),
        node_2=Node(label="Product", name="Macbook", properties={}),
        relationship="sells",
        metadata={"subject_gold": "APPLE_TECH", "object_gold": "MACBOOK"},
        order=3
    ),

    # =========================
    # 2. Abbreviation Ambiguity
    # =========================
    Edge(
        node_1=Node(label="GeopoliticalEntity", name="US", properties={}),
        node_2=Node(label="GeopoliticalEntity", name="Iraq", properties={}),
        relationship="invaded",
        metadata={"subject_gold": "UNITED_STATES", "object_gold": "IRAQ"},
        order=4
    ),
    Edge(
        node_1=Node(label="Person", name="US", properties={}),
        node_2=Node(label="SportsTournament", name="US Open", properties={}),
        relationship="won",
        metadata={"subject_gold": "US_TENNIS_PLAYER", "object_gold": "US_OPEN_TOURNAMENT"},
        order=5
    ),
    Edge(
        node_1=Node(label="GeopoliticalEntity", name="United States", properties={}),
        node_2=Node(label="City", name="Washington DC", properties={}),
        relationship="has capital",
        metadata={"subject_gold": "UNITED_STATES", "object_gold": "WASHINGTON_DC"},
        order=6
    ),

    # =========================
    # 3. Semantic Embedding Trap
    # =========================
    Edge(
        node_1=Node(label="Organization", name="Amazon", properties={}),
        node_2=Node(label="Product", name="Alexa", properties={}),
        relationship="develops",
        metadata={"subject_gold": "AMAZON_COMPANY", "object_gold": "ALEXA"},
        order=7
    ),
    Edge(
        node_1=Node(label="Organization", name="Amazon", properties={}),
        node_2=Node(label="City", name="Seattle", properties={}),
        relationship="headquartered in",
        metadata={"subject_gold": "AMAZON_COMPANY", "object_gold": "SEATTLE"},
        order=8
    ),
    Edge(
        node_1=Node(label="River", name="Amazon River", properties={}),
        node_2=Node(label="Country", name="Brazil", properties={}),
        relationship="flows through",
        metadata={"subject_gold": "AMAZON_RIVER", "object_gold": "BRAZIL"},
        order=9
    ),

    # =========================
    # 4. Role vs Entity Confusion
    # =========================
    Edge(
        node_1=Node(label="Role", name="President", properties={}),
        node_2=Node(label="GeopoliticalEntity", name="United States", properties={}),
        relationship="leads",
        metadata={"subject_gold": "US_PRESIDENT_ROLE", "object_gold": "UNITED_STATES"},
        order=10
    ),
    Edge(
        node_1=Node(label="Person", name="Joe Biden", properties={}),
        node_2=Node(label="Role", name="President", properties={}),
        relationship="is",
        metadata={"subject_gold": "JOE_BIDEN", "object_gold": "US_PRESIDENT_ROLE"},
        order=11
    ),
    Edge(
        node_1=Node(label="Person", name="Joe Biden", properties={}),
        node_2=Node(label="City", name="Scranton", properties={}),
        relationship="born in",
        metadata={"subject_gold": "JOE_BIDEN", "object_gold": "SCRANTON_PA"},
        order=12
    ),

    # =========================
    # 5. Predicate Paraphrase
    # =========================
    Edge(
        node_1=Node(label="Person", name="Elon Musk", properties={}),
        node_2=Node(label="Organization", name="SpaceX", properties={}),
        relationship="leads",
        metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
        order=13
    ),
    Edge(
        node_1=Node(label="Person", name="Elon Musk", properties={}),
        node_2=Node(label="Organization", name="SpaceX", properties={}),
        relationship="is CEO of",
        metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
        order=14
    ),
    Edge(
        node_1=Node(label="Person", name="Elon Musk", properties={}),
        node_2=Node(label="Organization", name="SpaceX", properties={}),
        relationship="runs",
        metadata={"subject_gold": "ELON_MUSK", "object_gold": "SPACEX"},
        order=15
    ),

    # =========================
    # 6. Predicate Polarity Conflict
    # =========================
    Edge(
        node_1=Node(label="Organization", name="Company X", properties={}),
        node_2=Node(label="Organization", name="Company Y", properties={}),
        relationship="acquired",
        metadata={"subject_gold": "COMPANY_X", "object_gold": "COMPANY_Y"},
        order=16
    ),
    Edge(
        node_1=Node(label="Organization", name="Company Y", properties={}),
        node_2=Node(label="Organization", name="Company X", properties={}),
        relationship="acquired",
        metadata={"subject_gold": "COMPANY_Y", "object_gold": "COMPANY_X"},
        order=17
    ),

    # =========================
    # 7. Structural / Neighborhood Deception
    # =========================
    Edge(
        node_1=Node(label="City", name="Paris", properties={}),
        node_2=Node(label="Country", name="France", properties={}),
        relationship="located in",
        metadata={"subject_gold": "PARIS_FRANCE", "object_gold": "FRANCE"},
        order=18
    ),
    Edge(
        node_1=Node(label="City", name="Paris", properties={}),
        node_2=Node(label="River", name="Seine", properties={}),
        relationship="has river",
        metadata={"subject_gold": "PARIS_FRANCE", "object_gold": "SEINE_RIVER"},
        order=19
    ),
    Edge(
        node_1=Node(label="City", name="Paris", properties={}),
        node_2=Node(label="State", name="Texas", properties={}),
        relationship="located in",
        metadata={"subject_gold": "PARIS_TEXAS", "object_gold": "TEXAS"},
        order=20
    ),
    Edge(
        node_1=Node(label="City", name="Paris", properties={}),
        node_2=Node(label="NumericValue", name="25000", properties={}),
        relationship="has population",
        metadata={"subject_gold": "PARIS_TEXAS", "object_gold": "POPULATION_25000"},
        order=21
    ),

    # =========================
    # 8. Copycat Organization
    # =========================
    Edge(
        node_1=Node(label="Organization", name="OpenAI", properties={}),
        node_2=Node(label="Product", name="ChatGPT", properties={}),
        relationship="develops",
        metadata={"subject_gold": "OPENAI", "object_gold": "CHATGPT"},
        order=22
    ),
    Edge(
        node_1=Node(label="Organization", name="OpenAI Research", properties={}),
        node_2=Node(label="Product", name="ChatGPT", properties={}),
        relationship="develops",
        metadata={"subject_gold": "OPENAI_RESEARCH", "object_gold": "CHATGPT"},
        order=23
    ),
    Edge(
        node_1=Node(label="Organization", name="OpenAI Research", properties={}),
        node_2=Node(label="Document", name="Paper A", properties={}),
        relationship="published",
        metadata={"subject_gold": "OPENAI_RESEARCH", "object_gold": "PAPER_A"},
        order=24
    ),
    Edge(
        node_1=Node(label="Organization", name="OpenAI", properties={}),
        node_2=Node(label="Document", name="Paper B", properties={}),
        relationship="published",
        metadata={"subject_gold": "OPENAI", "object_gold": "PAPER_B"},
        order=25
    ),

    # =========================
    # 9. Temporal Drift
    # =========================
    Edge(
        node_1=Node(label="Organization", name="Google", properties={}),
        node_2=Node(label="Person", name="Larry Page", properties={}),
        relationship="CEO",
        metadata={"subject_gold": "GOOGLE", "object_gold": "LARRY_PAGE"},
        order=26
    ),
    Edge(
        node_1=Node(label="Organization", name="Google", properties={}),
        node_2=Node(label="Person", name="Sundar Pichai", properties={}),
        relationship="CEO",
        metadata={"subject_gold": "GOOGLE", "object_gold": "SUNDAR_PICHAI"},
        order=27
    ),

    # =========================
    # 10. Compound Adversarial Case
    # =========================
    Edge(
        node_1=Node(label="Organization", name="Meta", properties={}),
        node_2=Node(label="Organization", name="Facebook", properties={}),
        relationship="formerly known as",
        metadata={"subject_gold": "META", "object_gold": "FACEBOOK"},
        order=28
    ),
    Edge(
        node_1=Node(label="Organization", name="Facebook", properties={}),
        node_2=Node(label="Organization", name="Instagram", properties={}),
        relationship="owns",
        metadata={"subject_gold": "FACEBOOK", "object_gold": "INSTAGRAM"},
        order=29
    ),
    Edge(
        node_1=Node(label="Organization", name="Meta AI", properties={}),
        node_2=Node(label="Model", name="LLaMA", properties={}),
        relationship="develops",
        metadata={"subject_gold": "META_AI", "object_gold": "LLAMA_MODEL"},
        order=30
    ),
    Edge(
        node_1=Node(label="Organization", name="Facebook AI Research", properties={}),
        node_2=Node(label="Model", name="LLaMA", properties={}),
        relationship="develops",
        metadata={"subject_gold": "FAIR", "object_gold": "LLAMA_MODEL"},
        order=31
    )
]

toy_data = []
unique_entities = set()
for edge in list_of_edges:
    unique_entities.add(edge.node_1.name)
    unique_entities.add(edge.node_2.name)

    toy_data.append({
        "subject": edge.node_1.name,
        "subject_gold": edge.metadata.get("subject_gold"),
        "predicate": edge.relationship,
        "object": edge.node_2.name,     
        "object_gold": edge.metadata.get("object_gold"),
        # "context": edge.metadata.get("summary"),
    })

In [4]:
# 1. Initialize predicate vocab for relational signatures
# pred2idx = {"works_at":0, "owns":1, "located_in":2}
pred2idx = {}

# 2. Initialize resolver
resolver = ClusterER(pred2idx=pred2idx, sim_threshold=0.65)

# 3. Resolve entities
for triple in toy_data:
    subj, pred, obj = triple["subject"], triple["predicate"], triple["object"]
    subj_id, _ = resolver.resolve_entity(subj, predicate=pred)
    obj_id, _ = resolver.resolve_entity(obj, predicate=pred, other_entity_id=subj_id)

# 4. Evaluate
resolver.evaluate(toy_data)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== Evaluation Results ===
Precision: 0.9524
Recall   : 0.9524
F1       : 0.9524

=== Cluster Alignment ===
Predicted Cluster: C5
Entities: {'Apple'}
Mixed cluster matches:
  - APPLE_FRUIT: {'Apple'}
  - APPLE_TECH: {'Apple'}
Predicted Cluster: C2
Entities: {'iPhone 15'}
Pure match with gold cluster: IPHONE_15
Predicted Cluster: C4
Entities: {'September'}
Pure match with gold cluster: MONTH_SEPTEMBER
Predicted Cluster: C6
Entities: {'Fruit'}
Pure match with gold cluster: FRUIT
Predicted Cluster: C7
Entities: {'Apple Inc.'}
Pure match with gold cluster: APPLE_TECH
Predicted Cluster: C8
Entities: {'Macbook'}
Pure match with gold cluster: MACBOOK
Predicted Cluster: C11
Entities: {'US'}
Mixed cluster matches:
  - US_TENNIS_PLAYER: {'US'}
  - UNITED_STATES: {'US'}
Predicted Cluster: C10
Entities: {'Iraq'}
Pure match with gold cluster: IRAQ
Predicted Cluster: C12
Entities: {'US Open'}
Pure match with gold cluster: US_OPEN_TOURNAMENT
Predicted Cluster: C22
Entities: {'United States'}
Pure ma

(0.9523809523786848, 0.9523809523786848, 0.9523809523286848)